# S01 — AWS CLI Setup on Ubuntu

This notebook installs AWS CLI v2 on Ubuntu, configures authentication, sets `us-east-1` as the only region, verifies the active identity, and works with Amazon S3 buckets.

> **Security:** Never paste an AWS secret access key into a notebook cell, source file, chat, or Git repository. Run `aws configure` in a terminal and enter credentials only at its hidden prompts. Prefer temporary credentials/SSO for production use; access keys are used here because they are the preferred course workflow.

## 1. Prerequisites

You need a 64-bit x86 Ubuntu machine, `sudo` access, an AWS account, and permission to use STS and S3. All commands below use Ubuntu's Bash shell and `apt`.

In [ ]:
%%bash
set -euo pipefail
uname -a
uname -m

## 2. Install AWS CLI v2

The following installs the official AWS CLI v2 bundle on 64-bit x86 Ubuntu.

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo apt-get install -y curl unzip
tmp_dir="$(mktemp -d)"
trap 'rm -rf "$tmp_dir"' EXIT
curl -fsSL "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "$tmp_dir/awscliv2.zip"
unzip -q "$tmp_dir/awscliv2.zip" -d "$tmp_dir"
sudo "$tmp_dir/aws/install" --update
aws --version

Restart the notebook kernel or open a new Ubuntu terminal if `aws` is not immediately found after installation.

## 3. Authentication option A — access key (course-preferred)

In the AWS console, create or select an IAM user with only the permissions required for this exercise. Create an access key only if your organization permits long-lived keys. Download or copy the secret once and store it in a password manager.

Run the next command **in a Linux terminal**, not as a notebook cell, so credentials do not become notebook output or history:

```bash
aws configure --profile course
```

Enter the access key ID, secret access key, default region `us-east-1`, and output format `json`. This writes credentials to `~/.aws/credentials` and settings to `~/.aws/config`. Do not commit either file.

In [ ]:
%%bash
# Inspect profile names and non-secret configuration only.
aws configure list-profiles
aws configure list --profile course

### Optional: temporary session credentials

If AWS gives you a three-part temporary credential, configure the access key and secret through `aws configure`, then add the session token from a terminal:

```bash
aws configure set aws_session_token 'YOUR_SESSION_TOKEN' --profile course
```

Do not place the real token in this notebook. Temporary credentials expire and must then be refreshed.

## 4. Authentication option B — browser login with IAM Identity Center (SSO)

Use this when your organization provides an AWS access portal URL and SSO region. Configure once, then log in through the browser:

```bash
aws configure sso --profile course-sso
aws sso login --profile course-sso
```

If the machine has no browser, use `aws sso login --profile course-sso --use-device-code` and open the displayed URL on another device. To end the cached SSO sessions, run `aws sso logout`.

## 5. Select a profile and set the region

This notebook uses the access-key profile named `course`. Exporting `AWS_PROFILE` makes subsequent CLI commands use it. Environment variables apply only to the current shell/cell process, so the examples also pass `--profile` explicitly when clarity matters.

In [ ]:
%%bash
set -euo pipefail
PROFILE=course
REGION=us-east-1
aws configure set region "$REGION" --profile "$PROFILE"
aws configure set output json --profile "$PROFILE"
aws configure get region --profile "$PROFILE"

## 6. Verify authentication

Always verify the account and principal before creating resources. This helps prevent changes in the wrong AWS account.

In [ ]:
%%bash
set -euo pipefail
aws sts get-caller-identity --profile course
aws configure list --profile course

## 7. List S3 buckets and objects

`aws s3` provides convenient high-level commands; `aws s3api` exposes the underlying S3 API.

In [ ]:
%%bash
# List all buckets in the account.
aws s3 ls --profile course

# Return selected bucket fields as a table.
aws s3api list-buckets \
  --profile course \
  --query 'Buckets[].{Name:Name,Created:CreationDate}' \
  --output table

# After setting BUCKET_NAME, list its top level or all objects recursively.
# aws s3 ls "s3://$BUCKET_NAME/" --profile course
# aws s3 ls "s3://$BUCKET_NAME/" --recursive --human-readable --summarize --profile course

## 8. Create an S3 bucket

S3 bucket names are globally unique across AWS, lowercase, and 3–63 characters. The following generates a likely-unique name from the account ID and current UTC time. Review the printed account, region, and bucket name before running the creation cell.

For `us-east-1`, `aws s3api create-bucket` must omit `--create-bucket-configuration`. The high-level `aws s3 mb` command below handles this correctly.

In [ ]:
%%bash
set -euo pipefail
PROFILE=course
REGION=us-east-1
ACCOUNT_ID="$(aws sts get-caller-identity --profile "$PROFILE" --query Account --output text)"
BUCKET_NAME="dataeng-${ACCOUNT_ID}-$(date -u +%Y%m%d%H%M%S)"
printf 'Account: %s\nRegion: %s\nBucket: %s\n' "$ACCOUNT_ID" "$REGION" "$BUCKET_NAME"
echo "Copy the bucket name into the creation cell below."

In [ ]:
%%bash
set -euo pipefail
PROFILE=course
REGION=us-east-1
BUCKET_NAME=replace-with-the-generated-unique-name

if [[ "$BUCKET_NAME" == replace-* ]]; then
  echo 'Set BUCKET_NAME before running this cell.' >&2
  exit 1
fi

aws s3 mb "s3://$BUCKET_NAME" --region "$REGION" --profile "$PROFILE"
aws s3api wait bucket-exists --bucket "$BUCKET_NAME" --profile "$PROFILE"
aws s3api get-bucket-location --bucket "$BUCKET_NAME" --profile "$PROFILE"
aws s3 ls --profile "$PROFILE"

### Recommended bucket safeguards

New S3 buckets have Block Public Access enabled by default. Explicitly enforce it and enable versioning for this exercise. Default encryption is also applied automatically by S3; you can inspect it with `get-bucket-encryption`.

In [ ]:
%%bash
set -euo pipefail
PROFILE=course
BUCKET_NAME=replace-with-your-bucket-name

if [[ "$BUCKET_NAME" == replace-* ]]; then
  echo 'Set BUCKET_NAME before running this cell.' >&2
  exit 1
fi

aws s3api put-public-access-block \
  --bucket "$BUCKET_NAME" \
  --public-access-block-configuration \
'BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true' \
  --profile "$PROFILE"

aws s3api put-bucket-versioning \
  --bucket "$BUCKET_NAME" \
  --versioning-configuration Status=Enabled \
  --profile "$PROFILE"

aws s3api get-public-access-block --bucket "$BUCKET_NAME" --profile "$PROFILE"
aws s3api get-bucket-versioning --bucket "$BUCKET_NAME" --profile "$PROFILE"
aws s3api get-bucket-encryption --bucket "$BUCKET_NAME" --profile "$PROFILE"

## 9. Useful S3 commands

Replace the placeholders before running these commands.

```bash
# Upload a file
aws s3 cp ./sample.csv s3://BUCKET_NAME/raw/sample.csv --profile course

# Download a file
aws s3 cp s3://BUCKET_NAME/raw/sample.csv ./sample.csv --profile course

# Synchronize a local directory to S3 (preview first)
aws s3 sync ./data/ s3://BUCKET_NAME/data/ --dryrun --profile course
aws s3 sync ./data/ s3://BUCKET_NAME/data/ --profile course

# Show an object's metadata
aws s3api head-object --bucket BUCKET_NAME --key raw/sample.csv --profile course

# Delete one object
aws s3 rm s3://BUCKET_NAME/raw/sample.csv --profile course
```

## 10. Cleanup (optional)

An empty bucket can be removed with `aws s3 rb`. A versioned bucket may retain object versions and delete markers even when it looks empty, so deleting it can require additional cleanup. Carefully confirm the exact bucket name before removing anything.

```bash
aws s3 ls s3://BUCKET_NAME --recursive --profile course
aws s3 rb s3://BUCKET_NAME --profile course
```

Avoid `aws s3 rb ... --force` unless you intentionally want to delete every current object in that bucket.

## Troubleshooting

- **Unable to locate credentials:** run `aws configure --profile course`, or log in again with `aws sso login --profile course-sso`.
- **ExpiredToken:** refresh temporary credentials or the SSO session.
- **AccessDenied:** the active IAM principal lacks the required action; inspect it with `aws sts get-caller-identity`.
- **BucketAlreadyExists:** choose another globally unique bucket name.
- **IllegalLocationConstraintException:** ensure the CLI `--region` and bucket location agree.
- **Clock skew/signature error:** enable Linux time synchronization, for example `sudo timedatectl set-ntp true`.